In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import Tool  # Correct modern import

# 🔐 Load API keys from .env
load_dotenv(dotenv_path=".env")

# 🔸 Step 1: Initialize Gemini 2.5 Flash
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0
)

# 🔸 Step 2: Define the QA Tool
qa_prompt = ChatPromptTemplate.from_template("Answer clearly: {question}")
qa_chain = qa_prompt | llm | StrOutputParser()

qa_tool = Tool(
    name="Simple_QA",
    func=qa_chain.invoke, # Use invoke for LCEL
    description="Answer factual questions clearly"
)

# 🔸 Step 3: Define the Summarizer Tool
summary_prompt = ChatPromptTemplate.from_template("Summarize the following text briefly:\n\n{text}")
summary_chain = summary_prompt | llm | StrOutputParser()

summary_tool = Tool(
    name="Summarizer",
    func=summary_chain.invoke,
    description="Summarizes long text into a concise version"
)

# 🚀 Step 4: Tool Usage Examples
qa_query = "What is LangGraph in LangChain?"
summary_text = """
LangGraph is a library built on top of LangChain that helps developers create stateful, multi-step agents 
as graphs. Each node represents a step like calling an LLM or a tool. It's ideal for advanced AI workflows.
"""

# 🖨️ Execution and Output
print("--- TOOL RESULTS ---")

# QA Tool Output
# Note: Since the prompt uses {question}, we pass that key in the dict
print("\n🧠 QA Tool Result:")
print(qa_tool.run({"question": qa_query}))

# Summarizer Tool Output
# Note: Since the prompt uses {text}, we pass that key in the dict
print("\n📝 Summarizer Tool Result:")
print(summary_tool.run({"text": summary_text}))

--- TOOL RESULTS ---

🧠 QA Tool Result:
LangGraph is a library built **on top of LangChain** that provides a more robust and flexible way to build **stateful, multi-actor, and cyclical applications** with large language models (LLMs).

Here's a breakdown of what that means and its relationship to LangChain:

1.  **LangChain as the Foundation:** LangChain is an orchestration framework for developing LLM-powered applications. It provides a standard interface for LLMs, tools, agents, chains, retrievers, and more. It helps you string together various components in a generally linear fashion (e.g., fetch data, ask LLM, use tool, return answer).

2.  **LangGraph for Advanced Control Flow and State:** While LangChain is excellent for many use cases, its standard `Chains` and `Agents` can sometimes be limited when you need:
    *   **Cyclical Execution:** An agent needs to try something, evaluate the result, and if it fails or needs more information, go back and try a different approach (e.g.,